# Camada Bronze — SISCAN / 10ª CRS

Ingestão dos arquivos CSV do TABNET/SISCAN (2020–2025) em nove tabelas Delta. Preserva cabeçalhos, valores brutos, linha Total e metadados de origem; a limpeza pertence à Silver.

## 1. Configuração e arquivos de origem

O dataset fica no volume `siscan_files_bronze`. Por padrão, o diretório existente é reutilizado para não sobrescrever arquivos já disponibilizados. Para substituir intencionalmente o conjunto pelo conteúdo de `dataset.zip`, configure `SUBSTITUIR_DATASET = True`.

A substituição é uma operação explícita e mantém uma cópia de segurança do diretório anterior no volume.

In [0]:
import csv
import io
import re
import shutil
import stat
import tempfile
import zipfile
from dataclasses import dataclass
from functools import reduce
from pathlib import Path, PurePosixPath

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

CATALOG = "`tcc-analise-siscan-10crs`"
SCHEMA_BRONZE = "bronze"
VOLUME_PATH = Path("/Volumes/tcc-analise-siscan-10crs/bronze/siscan_files_bronze")
ZIP_PATH = VOLUME_PATH / "dataset.zip"
PATH_RAW = VOLUME_PATH / "dataset"

SUBSTITUIR_DATASET = False
VALIDAR_REGRESSAO = True

REQUIRED_FOLDERS = (
    "q1_q2_exames_geral_2020_2025",
    "Q3_Faixa_Etaria",
    "Q4_Adequabilidade",
    "Q5_Resultados_Citopatologicos",
    "Q02-outubroRosa",
)
COL_MUNICIPIO = "Munic.de residencia"
COLS_ORIGEM = ("_file_source", "_ingestion_time")


def validate_dataset(root: Path) -> None:
    missing = [
        folder for folder in REQUIRED_FOLDERS
        if not (root / folder).is_dir() or not any((root / folder).glob("*.csv"))
    ]
    if missing:
        raise ValueError(f"Diretórios sem CSVs obrigatórios: {', '.join(missing)}")


def extract_archive(archive: Path, destination: Path) -> None:
    with zipfile.ZipFile(archive) as source:
        for member in source.infolist():
            member_path = PurePosixPath(member.filename)
            mode = member.external_attr >> 16
            if (
                member_path.is_absolute()
                or ".." in member_path.parts
                or "\\" in member.filename
                or stat.S_ISLNK(mode)
            ):
                raise ValueError(f"Entrada insegura no ZIP: {member.filename}")
        source.extractall(destination)


def prepare_dataset() -> Path:
    if PATH_RAW.exists() and not SUBSTITUIR_DATASET:
        validate_dataset(PATH_RAW)
        print(f"Dataset existente reutilizado: {PATH_RAW}")
        return PATH_RAW

    if not ZIP_PATH.is_file():
        raise FileNotFoundError(f"Arquivo não encontrado: {ZIP_PATH}")

    with tempfile.TemporaryDirectory(prefix="bronze_", dir=str(VOLUME_PATH)) as staging:
        extracted = Path(staging) / "extracted"
        extracted.mkdir()
        extract_archive(ZIP_PATH, extracted)
        nested = extracted / "dataset"
        source = nested if nested.is_dir() else extracted
        validate_dataset(source)

        backup = None
        if PATH_RAW.exists():
            backup_root = Path(tempfile.mkdtemp(prefix="dataset_backup_", dir=str(VOLUME_PATH)))
            backup = backup_root / "dataset"
            shutil.move(str(PATH_RAW), str(backup))
        try:
            shutil.move(str(source), str(PATH_RAW))
        except Exception:
            if PATH_RAW.exists():
                shutil.rmtree(PATH_RAW)
            if backup is not None and backup.exists():
                shutil.move(str(backup), str(PATH_RAW))
            raise

        if backup is not None:
            print(f"Dataset anterior preservado em: {backup}")

    print(f"Dataset extraído: {PATH_RAW}")
    return PATH_RAW


spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_BRONZE}")
prepare_dataset()

## 2. Leitura dos CSVs

A leitura individual preserva os cabeçalhos de cada extração. O TABNET pode omitir colunas sem registros; a união pelo nome mantém essas ausências como NULL.

Os campos provenientes do CSV são lidos como STRING, sem inferência automática de tipos. O encoding é identificado entre UTF-8 e ISO-8859-1. Os metadados técnicos são acrescentados sem normalização dos dados da fonte.

In [0]:
@dataclass(frozen=True)
class CsvSource:
    path: Path
    encoding: str
    header: tuple[str, ...]


@dataclass(frozen=True)
class BronzeTable:
    name: str
    sources: tuple[CsvSource, ...]


def table_name(name: str) -> str:
    return f"{CATALOG}.{SCHEMA_BRONZE}.{name}"


def quoted_col(name: str) -> Column:
    return F.col("`" + name.replace("`", "``") + "`")


def inspect_csv(path: Path) -> CsvSource:
    raw = path.read_bytes()
    try:
        decoded = raw.decode("utf-8-sig")
        encoding = "UTF-8"
    except UnicodeDecodeError:
        decoded = raw.decode("latin-1")
        encoding = "ISO-8859-1"

    reader = csv.reader(io.StringIO(decoded), delimiter=";")
    header = tuple(next(reader, ()))
    if not header or COL_MUNICIPIO not in header:
        raise ValueError(f"Cabeçalho inválido: {path.name}")
    if any(not name.strip() for name in header):
        raise ValueError(f"Coluna sem nome: {path.name}")
    if len({name.casefold() for name in header}) != len(header):
        raise ValueError(f"Colunas duplicadas: {path.name}")
    if {name.casefold() for name in header} & set(COLS_ORIGEM):
        raise ValueError(f"Cabeçalho usa nomes reservados: {path.name}")
    return CsvSource(path, encoding, header)


def list_sources(folder: str, pattern: str = "*.csv") -> tuple[CsvSource, ...]:
    paths = sorted(path for path in (PATH_RAW / folder).glob(pattern) if path.is_file())
    if not paths:
        raise FileNotFoundError(f"Nenhum arquivo encontrado: {folder}/{pattern}")
    return tuple(inspect_csv(path) for path in paths)


def read_csv(source: CsvSource) -> DataFrame:
    df = (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("delimiter", ";")
        .option("encoding", source.encoding)
        .option("mode", "FAILFAST")
        .load(str(source.path))
    )
    if tuple(df.columns) != source.header:
        raise ValueError(f"Cabeçalho interpretado pelo Spark difere da fonte: {source.path.name}")
    return (
        df.withColumn("_ingestion_time", F.current_timestamp())
        .withColumn("_file_source", F.col("_metadata.file_path"))
    )


def build_dataframe(spec: BronzeTable) -> DataFrame:
    if not spec.sources:
        raise ValueError(f"Nenhum CSV para {spec.name}")
    headers = {}
    for source in spec.sources:
        for name in source.header:
            key = name.casefold()
            if key in headers and headers[key] != name:
                raise ValueError(f"{spec.name}: cabeçalhos ambíguos: {headers[key]!r}, {name!r}")
            headers[key] = name

    frames = [read_csv(source) for source in spec.sources]
    return reduce(lambda left, right: left.unionByName(right, allowMissingColumns=True), frames)


base_tables = [
    BronzeTable("exames_geral_raw", list_sources("q1_q2_exames_geral_2020_2025")),
    BronzeTable("faixa_etaria_raw", list_sources("Q3_Faixa_Etaria")),
    BronzeTable("adequabilidade_raw", list_sources("Q4_Adequabilidade")),
    BronzeTable(
        "resultados_raw",
        list_sources("Q5_Resultados_Citopatologicos", "q5-siscan_colo_alterados_faixa_etaria_*.csv"),
    ),
]

## 2.1. Categorias citopatológicas

As extrações adicionais são separadas pela estrutura do cabeçalho: mensal, anual, faixa etária por categoria e consolidado de alterados. As bases não são misturadas com a tabela anual original.

In [0]:
def category_structure(source: CsvSource) -> str:
    values = [
        name.strip() for name in source.header
        if name != COL_MUNICIPIO and name.strip().casefold() != "total"
    ]
    if values and all(re.fullmatch(r"[^/]+/\d{4}", name) for name in values):
        return "mensal"
    if values and all(re.fullmatch(r"\d{4}", name) for name in values):
        return "anual"
    raise ValueError(f"Estrutura temporal desconhecida: {source.path.name}")


def category_tables() -> list[BronzeTable]:
    grouped: dict[str, list[CsvSource]] = {
        "mensal": [], "anual": [], "faixa_etaria": [], "consolidado": [],
    }
    for source in list_sources("Q5_Resultados_Citopatologicos"):
        name = source.path.name
        if name.startswith("q5-siscan_Exames_Alterados"):
            grouped["consolidado"].append(source)
        elif name.endswith("_anual.csv"):
            grouped[category_structure(source)].append(source)
        elif name.endswith("_faixa_etaria.csv"):
            grouped["faixa_etaria"].append(source)
        elif not name.startswith("q5-siscan_colo_alterados_faixa_etaria_"):
            print(f"CSV Q5 fora do mapeamento: {name}")

    return [
        BronzeTable("resultados_categoria_mensal_raw", tuple(grouped["mensal"])),
        BronzeTable("resultados_categoria_anual_raw", tuple(grouped["anual"])),
        BronzeTable("resultados_categoria_faixa_etaria_raw", tuple(grouped["faixa_etaria"])),
        BronzeTable("resultados_alterados_consolidado_raw", tuple(grouped["consolidado"])),
    ]


q5_tables = category_tables()

## 2.2. Outubro Rosa

Extrações mensais de setembro a dezembro de 2020 a 2025, por município e faixa etária. Os 24 arquivos são lidos individualmente e suas competências são conferidas antes da persistência.

In [0]:
def validate_campaign_sources(sources: tuple[CsvSource, ...]) -> None:
    expected = {
        (year, month)
        for year in range(2020, 2026)
        for month in ("SETEMBR", "OUTUBR", "NOVEM", "DEZEMB")
    }
    observed = set()
    for source in sources:
        match = re.fullmatch(
            r"(Setembr|Outubr|Novem|Dezemb)_(\d{4})(?:\s|%20)*\.csv",
            source.path.name,
            re.IGNORECASE,
        )
        if not match:
            raise ValueError(f"Nome de competência inválido: {source.path.name}")
        key = (int(match.group(2)), match.group(1).upper())
        if key in observed:
            raise ValueError(f"Competência duplicada: {source.path.name}")
        observed.add(key)
    if observed != expected:
        raise ValueError(
            f"Competências faltantes: {sorted(expected - observed)}; "
            f"fora do escopo: {sorted(observed - expected)}"
        )


campaign_sources = list_sources("Q02-outubroRosa")
validate_campaign_sources(campaign_sources)

tables = [
    *base_tables,
    *q5_tables,
    BronzeTable("outubro_rosa_faixa_etaria_raw", campaign_sources),
]

all_paths = [source.path for spec in tables for source in spec.sources]
if len(all_paths) != len(set(all_paths)):
    raise ValueError("Um mesmo arquivo foi atribuído a mais de uma tabela Bronze.")

## 3. Validação e persistência Delta

Todas as tabelas são preparadas e validadas antes da primeira gravação. A validação confere registros, arquivos de origem, linhagem e equivalência com os dados já publicados.

A regressão ignora apenas o timestamp de ingestão e compara os campos como STRING, incluindo a origem do arquivo. Divergências de valores ou cabeçalhos interrompem a carga. Desative `VALIDAR_REGRESSAO` somente após aprovar uma mudança intencional da fonte ou do encoding.

A substituição usa overwrite Delta com `overwriteSchema`, sem exclusão prévia de tabelas. A gravação é transacional por tabela, não de forma conjunta.

In [0]:
def validate_table(spec: BronzeTable, df: DataFrame) -> dict:
    stats = df.agg(
        F.count("*").alias("linhas"),
        F.countDistinct("_file_source").alias("arquivos"),
        F.count(F.when(F.col("_file_source").isNull(), 1)).alias("origem_nula"),
        F.count(F.when(F.col("_ingestion_time").isNull(), 1)).alias("ingestao_nula"),
    ).first().asDict()
    if (
        not stats["linhas"]
        or stats["arquivos"] != len(spec.sources)
        or stats["origem_nula"]
        or stats["ingestao_nula"]
    ):
        raise ValueError(f"{spec.name}: carga inválida — {stats}")

    target = table_name(spec.name)
    if VALIDAR_REGRESSAO and spark.catalog.tableExists(target):
        current = spark.table(target)
        columns = [name for name in df.columns if name != "_ingestion_time"]
        existing = set(current.columns) - {"_ingestion_time"}
        if existing != set(columns):
            raise ValueError(
                f"{spec.name}: alteração de cabeçalho; "
                f"removidos={sorted(existing - set(columns))}, "
                f"novos={sorted(set(columns) - existing)}"
            )

        def comparable(frame: DataFrame) -> DataFrame:
            return frame.select(*(quoted_col(name).cast("string").alias(name) for name in columns))

        before, after = comparable(current), comparable(df)
        removed = before.exceptAll(after).count()
        added = after.exceptAll(before).count()
        if removed or added:
            raise ValueError(
                f"{spec.name}: regressão — {removed} registros removidos/alterados, "
                f"{added} registros novos/alterados"
            )

    return stats


def write_table(spec: BronzeTable, df: DataFrame) -> None:
    (
        df.write.format("delta")
        .option("delta.columnMapping.mode", "name")
        .option("delta.minReaderVersion", "2")
        .option("delta.minWriterVersion", "5")
        .option("overwriteSchema", "true")
        .mode("overwrite")
        .saveAsTable(table_name(spec.name))
    )


prepared = [(spec, build_dataframe(spec)) for spec in tables]
metrics = {}
errors = []
for spec, df in prepared:
    try:
        metrics[spec.name] = validate_table(spec, df)
        stats = metrics[spec.name]
        print(f"{spec.name}: {stats['linhas']} linhas | {stats['arquivos']} arquivos | OK")
    except ValueError as error:
        errors.append(str(error))

if errors:
    raise ValueError("Falha na validação Bronze:\n" + "\n".join(errors))

for spec, df in prepared:
    write_table(spec, df)
    print(f"{spec.name}: {metrics[spec.name]['linhas']} linhas gravadas.")

## 4. Conferência da carga

Resumo das nove tabelas persistidas, com quantidade de arquivos e registros. A amostra mantém os campos brutos da fonte, inclusive a linha Total.

In [0]:
summary = []
for spec in tables:
    stats = spark.table(table_name(spec.name)).agg(
        F.count("*").alias("linhas"),
        F.countDistinct("_file_source").alias("arquivos"),
    ).first()
    if stats["linhas"] != metrics[spec.name]["linhas"] or stats["arquivos"] != len(spec.sources):
        raise ValueError(f"Falha na conferência após gravação: {spec.name}")
    summary.append((spec.name, stats["arquivos"], stats["linhas"]))

display(spark.createDataFrame(summary, "tabela string, arquivos long, linhas long").orderBy("tabela"))
display(spark.table(table_name("exames_geral_raw")).limit(20))